# GradX — Validation & Examples
Compares GradX implementations against scikit-learn and demonstrates training on MNIST.

## Linear Regression

In [4]:
import numpy as np
from sklearn.linear_model import LinearRegression, SGDRegressor

np.random.seed(42)
X = 2 * np.random.rand(100, 2)
w_true = np.array([[3], [3]])
y = 2 + X @ w_true + np.random.randn(100, 1)

test_X = np.array([[1, 1], [2, 2]])

In [5]:
# sklearn — Normal Equation
sklearn_lr = LinearRegression()
sklearn_lr.fit(X, y)
print("sklearn LinearRegression:", sklearn_lr.predict(test_X))

sklearn LinearRegression: [[ 8.11907867]
 [14.46588507]]


In [6]:
# sklearn — SGD
sgd_reg = SGDRegressor(max_iter=1000, learning_rate='constant', eta0=0.01)
sgd_reg.fit(X, y.ravel())
print("sklearn SGDRegressor:", sgd_reg.predict(test_X))

sklearn SGDRegressor: [ 8.17959948 14.46737386]


In [7]:
# GradX — Normal Equation
from linear_regression import LinearRegression as GradXLR

model_ne = GradXLR(method='normal_equation')
model_ne.fit(X, y)
print("GradX Normal Equation:", model_ne.predict(test_X))

Normal Equation - Final Loss: 14.393804
GradX Normal Equation: [ 8.11907867 14.46588507]


In [8]:
# GradX — Gradient Descent
model_gd = GradXLR(method='gradient_descent')
model_gd.fit(X, y, learning_rate=0.01, epochs=1000)
print("GradX Gradient Descent:", model_gd.predict(test_X))

Starting Loss: 70.352363
Final Loss: 0.993176
GradX Gradient Descent: [ 8.14068028 14.2312776 ]


## Polynomial Regression

In [10]:
import numpy as np
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression

np.random.seed(42)
X = np.linspace(-3, 3, 100).reshape(-1, 1)
y = 3 * X**2 + 2 * X + 1 + np.random.randn(100, 1) * 2

test_X = np.array([[-2], [0], [2]])

poly = PolynomialFeatures(degree=2, include_bias=True)
X_poly = poly.fit_transform(X)
test_X_poly = poly.transform(test_X)

sklearn_model = LinearRegression()
sklearn_model.fit(X_poly, y)
print("sklearn PolynomialRegression:", sklearn_model.predict(test_X_poly).ravel())

sklearn PolynomialRegression: [ 8.73860727  0.66766832 16.92251751]


In [11]:
from polynomial_regression import PolynomialRegression as GradXPR

gradx_poly = GradXPR(degree=2)
gradx_poly.fit(X, y, learning_rate=0.01, epochs=1000)
print("GradX PolynomialRegression:", gradx_poly.predict(test_X))

Starting Loss:184.809460
Final Loss:3.247196
GradX PolynomialRegression: [ 8.73827163  0.66635121 16.92218186]


## Neural Network on MNIST

In [13]:
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
import numpy as np

mnist = fetch_openml('mnist_784', version=1, as_frame=False)
X, y = mnist["data"], mnist["target"].astype(int)

X = X / 255.0

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=5000, train_size=50000, random_state=42
)

encoder = OneHotEncoder(sparse_output=False)
y_train_onehot = encoder.fit_transform(y_train.reshape(-1, 1))
y_test_onehot = encoder.transform(y_test.reshape(-1, 1))

print("X_train:", X_train.shape)
print("X_test: ", X_test.shape)

X_train: (50000, 784)
X_test:  (5000, 784)


In [14]:
from neural_network import NeuralNetwork

def evaluate(nn, X_test, y_test_onehot):
    y_pred = np.argmax(nn.predict_proba(X_test), axis=1)
    y_true = np.argmax(y_test_onehot, axis=1)
    return np.mean(y_pred == y_true)

In [15]:
# SGD
nn = NeuralNetwork(layer_sizes=[784, 64, 10], activation='relu', optimizer='sgd')
nn.fit(X_train, y_train_onehot, learning_rate=0.01, batch_size=32, epochs=20)
print("SGD Test Accuracy:", evaluate(nn, X_test, y_test_onehot))

Epoch 1/20
Batch 1563/1563 | Loss: 0.3738 | Avg Loss: 0.687726 | Accuracy: 0.8256
Epoch 2/20
Batch 1563/1563 | Loss: 0.4094 | Avg Loss: 0.354811 | Accuracy: 0.9013
Epoch 3/20
Batch 1563/1563 | Loss: 0.5115 | Avg Loss: 0.306512 | Accuracy: 0.9135
Epoch 4/20
Batch 1563/1563 | Loss: 0.0861 | Avg Loss: 0.278367 | Accuracy: 0.9218
Epoch 5/20
Batch 1563/1563 | Loss: 0.4570 | Avg Loss: 0.257182 | Accuracy: 0.9282
Epoch 6/20
Batch 1563/1563 | Loss: 0.1346 | Avg Loss: 0.239633 | Accuracy: 0.9332
Epoch 7/20
Batch 1563/1563 | Loss: 0.0615 | Avg Loss: 0.225312 | Accuracy: 0.9372
Epoch 8/20
Batch 1563/1563 | Loss: 0.2311 | Avg Loss: 0.213203 | Accuracy: 0.9404
Epoch 9/20
Batch 1563/1563 | Loss: 0.2019 | Avg Loss: 0.202074 | Accuracy: 0.9437
Epoch 10/20
Batch 1563/1563 | Loss: 0.1258 | Avg Loss: 0.192320 | Accuracy: 0.9462
Epoch 11/20
Batch 1563/1563 | Loss: 0.1431 | Avg Loss: 0.183101 | Accuracy: 0.9484
Epoch 12/20
Batch 1563/1563 | Loss: 0.4233 | Avg Loss: 0.175144 | Accuracy: 0.9507
Epoch 13/20
B

In [16]:
# Adam
nn = NeuralNetwork(layer_sizes=[784, 64, 10], activation='relu', optimizer='adam')
nn.fit(X_train, y_train_onehot, learning_rate=0.01, batch_size=32, epochs=20)
print("Adam Test Accuracy:", evaluate(nn, X_test, y_test_onehot))

Epoch 1/20
Batch 1563/1563 | Loss: 0.1864 | Avg Loss: 0.252598 | Accuracy: 0.9244
Epoch 2/20
Batch 1563/1563 | Loss: 0.0101 | Avg Loss: 0.167619 | Accuracy: 0.9519
Epoch 3/20
Batch 1563/1563 | Loss: 0.0007 | Avg Loss: 0.146191 | Accuracy: 0.9583
Epoch 4/20
Batch 1563/1563 | Loss: 0.2348 | Avg Loss: 0.131471 | Accuracy: 0.9619
Epoch 5/20
Batch 1563/1563 | Loss: 0.0474 | Avg Loss: 0.121942 | Accuracy: 0.9661
Epoch 6/20
Batch 1563/1563 | Loss: 0.9108 | Avg Loss: 0.114771 | Accuracy: 0.9692
Epoch 7/20
Batch 1563/1563 | Loss: 0.1570 | Avg Loss: 0.113807 | Accuracy: 0.9690
Epoch 8/20
Batch 1563/1563 | Loss: 0.0601 | Avg Loss: 0.108561 | Accuracy: 0.9719
Epoch 9/20
Batch 1563/1563 | Loss: 0.0000 | Avg Loss: 0.098000 | Accuracy: 0.9746
Epoch 10/20
Batch 1563/1563 | Loss: 0.0000 | Avg Loss: 0.096775 | Accuracy: 0.9746
Epoch 11/20
Batch 1563/1563 | Loss: 0.0057 | Avg Loss: 0.094377 | Accuracy: 0.9753
Epoch 12/20
Batch 1563/1563 | Loss: 0.5381 | Avg Loss: 0.089983 | Accuracy: 0.9764
Epoch 13/20
B

In [17]:

nn = NeuralNetwork(layer_sizes=[784, 64, 10], activation='relu', optimizer='nadam')
nn.fit(X_train, y_train_onehot, learning_rate=0.01, batch_size=32, epochs=20)
print("NAdam Test Accuracy:", evaluate(nn, X_test, y_test_onehot))

Epoch 1/20
Batch 1563/1563 | Loss: 0.1950 | Avg Loss: 0.245084 | Accuracy: 0.9267
Epoch 2/20
Batch 1563/1563 | Loss: 0.0127 | Avg Loss: 0.162396 | Accuracy: 0.9519
Epoch 3/20
Batch 1563/1563 | Loss: 0.0017 | Avg Loss: 0.143331 | Accuracy: 0.9591
Epoch 4/20
Batch 1563/1563 | Loss: 0.0229 | Avg Loss: 0.126279 | Accuracy: 0.9648
Epoch 5/20
Batch 1563/1563 | Loss: 0.0672 | Avg Loss: 0.128009 | Accuracy: 0.9647
Epoch 6/20
Batch 1563/1563 | Loss: 0.5512 | Avg Loss: 0.110993 | Accuracy: 0.9702
Epoch 7/20
Batch 1563/1563 | Loss: 0.0026 | Avg Loss: 0.112861 | Accuracy: 0.9698
Epoch 8/20
Batch 1563/1563 | Loss: 0.0110 | Avg Loss: 0.105725 | Accuracy: 0.9713
Epoch 9/20
Batch 1563/1563 | Loss: 0.0000 | Avg Loss: 0.097804 | Accuracy: 0.9733
Epoch 10/20
Batch 1563/1563 | Loss: 0.0149 | Avg Loss: 0.093461 | Accuracy: 0.9754
Epoch 11/20
Batch 1563/1563 | Loss: 0.0752 | Avg Loss: 0.090604 | Accuracy: 0.9763
Epoch 12/20
Batch 1563/1563 | Loss: 0.3911 | Avg Loss: 0.089294 | Accuracy: 0.9768
Epoch 13/20
B

In [18]:
# AdamW
nn = NeuralNetwork(layer_sizes=[784, 128, 64, 10], activation='relu', optimizer='adamw')
nn.fit(X_train, y_train_onehot, learning_rate=0.01, batch_size=32, epochs=20)
print("AdamW Test Accuracy:", evaluate(nn, X_test, y_test_onehot))

Epoch 1/20
Batch 1563/1563 | Loss: 0.3768 | Avg Loss: 0.273272 | Accuracy: 0.9194
Epoch 2/20
Batch 1563/1563 | Loss: 0.0419 | Avg Loss: 0.176788 | Accuracy: 0.9503
Epoch 3/20
Batch 1563/1563 | Loss: 0.0030 | Avg Loss: 0.159606 | Accuracy: 0.9554
Epoch 4/20
Batch 1563/1563 | Loss: 0.4694 | Avg Loss: 0.137990 | Accuracy: 0.9615
Epoch 5/20
Batch 1563/1563 | Loss: 0.1935 | Avg Loss: 0.132938 | Accuracy: 0.9638
Epoch 6/20
Batch 1563/1563 | Loss: 0.1317 | Avg Loss: 0.126102 | Accuracy: 0.9664
Epoch 7/20
Batch 1563/1563 | Loss: 0.0174 | Avg Loss: 0.121899 | Accuracy: 0.9668
Epoch 8/20
Batch 1563/1563 | Loss: 0.2579 | Avg Loss: 0.115213 | Accuracy: 0.9691
Epoch 9/20
Batch 1563/1563 | Loss: 0.1866 | Avg Loss: 0.115158 | Accuracy: 0.9696
Epoch 10/20
Batch 1563/1563 | Loss: 0.2116 | Avg Loss: 0.104003 | Accuracy: 0.9719
Epoch 11/20
Batch 1563/1563 | Loss: 0.0630 | Avg Loss: 0.099789 | Accuracy: 0.9736
Epoch 12/20
Batch 1563/1563 | Loss: 0.1412 | Avg Loss: 0.104292 | Accuracy: 0.9721
Epoch 13/20
B

In [19]:
# AdaGrad
nn = NeuralNetwork(layer_sizes=[784, 128, 64, 10], activation='relu', optimizer='adagrad')
nn.fit(X_train, y_train_onehot, learning_rate=0.01, batch_size=32, epochs=20)
print("AdaGrad Test Accuracy:", evaluate(nn, X_test, y_test_onehot))

Epoch 1/20
Batch 1563/1563 | Loss: 0.1073 | Avg Loss: 0.239104 | Accuracy: 0.9301
Epoch 2/20
Batch 1563/1563 | Loss: 0.0390 | Avg Loss: 0.129840 | Accuracy: 0.9629
Epoch 3/20
Batch 1563/1563 | Loss: 0.2677 | Avg Loss: 0.102047 | Accuracy: 0.9713
Epoch 4/20
Batch 1563/1563 | Loss: 0.0550 | Avg Loss: 0.085807 | Accuracy: 0.9759
Epoch 5/20
Batch 1563/1563 | Loss: 0.0340 | Avg Loss: 0.074239 | Accuracy: 0.9791
Epoch 6/20
Batch 1563/1563 | Loss: 0.0044 | Avg Loss: 0.065967 | Accuracy: 0.9824
Epoch 7/20
Batch 1563/1563 | Loss: 0.0527 | Avg Loss: 0.059900 | Accuracy: 0.9838
Epoch 8/20
Batch 1563/1563 | Loss: 0.0156 | Avg Loss: 0.054411 | Accuracy: 0.9855
Epoch 9/20
Batch 1563/1563 | Loss: 0.0475 | Avg Loss: 0.049722 | Accuracy: 0.9870
Epoch 10/20
Batch 1563/1563 | Loss: 0.0467 | Avg Loss: 0.046047 | Accuracy: 0.9879
Epoch 11/20
Batch 1563/1563 | Loss: 0.0029 | Avg Loss: 0.042704 | Accuracy: 0.9893
Epoch 12/20
Batch 1563/1563 | Loss: 0.0162 | Avg Loss: 0.040022 | Accuracy: 0.9900
Epoch 13/20
B

In [20]:
# AdaMax
nn = NeuralNetwork(layer_sizes=[784, 128, 64, 10], activation='relu', optimizer='adamax')
nn.fit(X_train, y_train_onehot, learning_rate=0.01, batch_size=32, epochs=20)
print("AdaMax Test Accuracy:", evaluate(nn, X_test, y_test_onehot))

Epoch 1/20
Batch 1563/1563 | Loss: 0.0925 | Avg Loss: 0.218729 | Accuracy: 0.9335
Epoch 2/20
Batch 1563/1563 | Loss: 0.0007 | Avg Loss: 0.099732 | Accuracy: 0.9697
Epoch 3/20
Batch 1563/1563 | Loss: 0.0144 | Avg Loss: 0.071832 | Accuracy: 0.9779
Epoch 4/20
Batch 1563/1563 | Loss: 0.3018 | Avg Loss: 0.053303 | Accuracy: 0.9831
Epoch 5/20
Batch 1563/1563 | Loss: 0.0169 | Avg Loss: 0.039322 | Accuracy: 0.9873
Epoch 6/20
Batch 1563/1563 | Loss: 0.0398 | Avg Loss: 0.029989 | Accuracy: 0.9901
Epoch 7/20
Batch 1563/1563 | Loss: 0.0007 | Avg Loss: 0.024693 | Accuracy: 0.9920
Epoch 8/20
Batch 1563/1563 | Loss: 0.0007 | Avg Loss: 0.017057 | Accuracy: 0.9945
Epoch 9/20
Batch 1563/1563 | Loss: 0.0183 | Avg Loss: 0.014448 | Accuracy: 0.9951
Epoch 10/20
Batch 1563/1563 | Loss: 0.0001 | Avg Loss: 0.009811 | Accuracy: 0.9968
Epoch 11/20
Batch 1563/1563 | Loss: 0.0057 | Avg Loss: 0.009696 | Accuracy: 0.9966
Epoch 12/20
Batch 1563/1563 | Loss: 0.0000 | Avg Loss: 0.007087 | Accuracy: 0.9978
Epoch 13/20
B

In [21]:
# RMSProp
nn = NeuralNetwork(layer_sizes=[784, 128, 64, 10], activation='relu', optimizer='rmsprop')
nn.fit(X_train, y_train_onehot, learning_rate=0.01, batch_size=32, epochs=20)
print("RMSProp Test Accuracy:", evaluate(nn, X_test, y_test_onehot))

Epoch 1/20
Batch 1563/1563 | Loss: 0.4750 | Avg Loss: 0.344635 | Accuracy: 0.9078
Epoch 2/20
Batch 1563/1563 | Loss: 0.1357 | Avg Loss: 0.269479 | Accuracy: 0.9407
Epoch 3/20
Batch 1563/1563 | Loss: 0.0023 | Avg Loss: 0.272914 | Accuracy: 0.9430
Epoch 4/20
Batch 1563/1563 | Loss: 0.2389 | Avg Loss: 0.267606 | Accuracy: 0.9464
Epoch 5/20
Batch 1563/1563 | Loss: 0.0000 | Avg Loss: 0.252286 | Accuracy: 0.9487
Epoch 6/20
Batch 1563/1563 | Loss: 0.3110 | Avg Loss: 0.254668 | Accuracy: 0.9500
Epoch 7/20
Batch 1563/1563 | Loss: 0.5057 | Avg Loss: 0.261968 | Accuracy: 0.9478
Epoch 8/20
Batch 1563/1563 | Loss: 0.0001 | Avg Loss: 0.283382 | Accuracy: 0.9425
Epoch 9/20
Batch 1563/1563 | Loss: 1.0671 | Avg Loss: 0.293780 | Accuracy: 0.9398
Epoch 10/20
Batch 1563/1563 | Loss: 1.3765 | Avg Loss: 0.297805 | Accuracy: 0.9384
Epoch 11/20
Batch 1563/1563 | Loss: 0.5055 | Avg Loss: 0.315404 | Accuracy: 0.9361
Epoch 12/20
Batch 1563/1563 | Loss: 0.0048 | Avg Loss: 0.335554 | Accuracy: 0.9305
Epoch 13/20
B

In [22]:
# Momentum with dropout
nn = NeuralNetwork(layer_sizes=[784, 128, 64, 10], activation='relu', optimizer='momentum', dropout=0.1)
nn.fit(X_train, y_train_onehot, learning_rate=0.01, batch_size=32, epochs=20)
print("Momentum Test Accuracy:", evaluate(nn, X_test, y_test_onehot))

Epoch 1/20
Batch 1563/1563 | Loss: 0.1281 | Avg Loss: 0.345911 | Accuracy: 0.8947
Epoch 2/20
Batch 1563/1563 | Loss: 0.3308 | Avg Loss: 0.161025 | Accuracy: 0.9518
Epoch 3/20
Batch 1563/1563 | Loss: 0.0915 | Avg Loss: 0.123250 | Accuracy: 0.9618
Epoch 4/20
Batch 1563/1563 | Loss: 0.0270 | Avg Loss: 0.101929 | Accuracy: 0.9693
Epoch 5/20
Batch 1563/1563 | Loss: 0.0049 | Avg Loss: 0.086118 | Accuracy: 0.9736
Epoch 6/20
Batch 1563/1563 | Loss: 0.0971 | Avg Loss: 0.071874 | Accuracy: 0.9780
Epoch 7/20
Batch 1563/1563 | Loss: 0.0077 | Avg Loss: 0.066617 | Accuracy: 0.9790
Epoch 8/20
Batch 1563/1563 | Loss: 0.0097 | Avg Loss: 0.060372 | Accuracy: 0.9812
Epoch 9/20
Batch 1563/1563 | Loss: 0.0022 | Avg Loss: 0.055142 | Accuracy: 0.9830
Epoch 10/20
Batch 1563/1563 | Loss: 0.0804 | Avg Loss: 0.048411 | Accuracy: 0.9842
Epoch 11/20
Batch 1563/1563 | Loss: 0.0833 | Avg Loss: 0.044068 | Accuracy: 0.9858
Epoch 12/20
Batch 1563/1563 | Loss: 0.0153 | Avg Loss: 0.042313 | Accuracy: 0.9858
Epoch 13/20
B

## GradX Modular API (gradx.nn)

In [24]:
import gradx.nn as nn
import gradx.optimizers as optim

model = nn.Sequential(
    nn.Linear(784, 256),
    nn.ReLU(),
    nn.Linear(256, 128),
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(128, 10)
)
print(model)

Sequential(
  (0): Linear(in_features=784, out_features=256, bias=True)
  (1): ReLU()
  (2): Linear(in_features=256, out_features=128, bias=True)
  (3): ReLU()
  (4): Dropout(p=0.3)
  (5): Linear(in_features=128, out_features=10, bias=True)
)


In [25]:
optimizer = optim.Adam(learning_rate=0.001)

num_epochs = 20
batch_size = 64

X_tr, y_tr = X_train.copy(), y_train.copy()

for epoch in range(num_epochs):
    perm = np.random.permutation(len(X_tr))
    X_tr, y_tr = X_tr[perm], y_tr[perm]

    total_loss = 0
    num_batches = 0

    model.train()
    for i in range(0, len(X_tr), batch_size):
        X_batch = X_tr[i:i + batch_size]
        y_batch = y_tr[i:i + batch_size]

        logits = model(X_batch)

        exps = np.exp(logits - np.max(logits, axis=1, keepdims=True))
        probs = exps / np.sum(exps, axis=1, keepdims=True)

        N = X_batch.shape[0]
        loss = -np.sum(np.log(np.clip(probs[np.arange(N), y_batch], 1e-7, 1.0))) / N
        total_loss += loss
        num_batches += 1

        grad_logits = probs.copy()
        grad_logits[np.arange(N), y_batch] -= 1
        grad_logits /= N

        model.backward(grad_logits)
        optimizer.step(model.parameters())

    print(f"Epoch {epoch+1}/{num_epochs}, Loss: {total_loss / num_batches:.4f}")

Epoch 1/20, Loss: 0.2926
Epoch 2/20, Loss: 0.1199
Epoch 3/20, Loss: 0.0867
Epoch 4/20, Loss: 0.0621
Epoch 5/20, Loss: 0.0566
Epoch 6/20, Loss: 0.0397
Epoch 7/20, Loss: 0.0323
Epoch 8/20, Loss: 0.0291
Epoch 9/20, Loss: 0.0244
Epoch 10/20, Loss: 0.0214
Epoch 11/20, Loss: 0.0189
Epoch 12/20, Loss: 0.0191
Epoch 13/20, Loss: 0.0171
Epoch 14/20, Loss: 0.0164
Epoch 15/20, Loss: 0.0161
Epoch 16/20, Loss: 0.0140
Epoch 17/20, Loss: 0.0115
Epoch 18/20, Loss: 0.0112
Epoch 19/20, Loss: 0.0109
Epoch 20/20, Loss: 0.0136


In [26]:
model.eval()
logits = model(X_test)
preds = np.argmax(logits, axis=1)
test_acc = np.mean(preds == y_test)
print(f"Test Accuracy: {test_acc * 100:.2f}%")

Test Accuracy: 97.66%
